# 2.1.4 심플렉스 알고리즘 — 그래픽·대수 대응 (Simplex: Graphic & Algebraic)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ffraile/operations-research-notebooks/blob/main/docs/source/CLP/tutorials/Simplex%20Graphic%20explanation.ipynb)

## 소개 (Introduction)

이 노트북에서는 <strong>심플렉스 알고리즘(Simplex Method)</strong>을 **그래픽 표현**과 <strong>대수적 태블로(Tableau)</strong>를 교차하며 분석합니다.

핵심 아이디어:
- 심플렉스는 가능 영역의 꼭짓점을 **체계적으로 이동**하며 최적해를 찾는 알고리즘
- 각 반복(Iteration)에서 목적 함수 값이 단조 증가 (최대화 문제 기준)
- 그래픽으로는 **꼭짓점 간 이동**, 대수적으로는 **기저 변수(Basic Variable) 교체**로 표현

## 문제 정의 (Problem Definition)

**생산 믹스(Production Mix) 문제**

어떤 기업이 두 종류의 제품 $P_1$, $P_2$를 생산합니다.

- $P_1$ 판매가: 300유로/개
- $P_2$ 판매가: 250유로/개

**자원 소비량:**
- $P_1$: 작업자 2시간 + 기계 1시간
- $P_2$: 작업자 1시간 + 기계 3시간

**자원 제약:**
- 작업자 가용 시간: 하루 40시간 이하
- 기계 가동 시간: 하루 45시간 이하
- 마케팅 제약: $P_1 \leq 12$개/일

**목표:** 하루 수익을 최대화하는 $P_1$, $P_2$ 생산량 결정

## 모델 (Model)

**목적 함수 (최대화):**

$$\max Z = 300x_{1} + 250x_{2}$$

**결정 변수:**
- $x_1$: $P_1$ 일일 생산량
- $x_2$: $P_2$ 일일 생산량

**제약 조건:**

$$2x_{1} + x_{2} \leq 40 \quad \text{(작업자 시간)}$$

$$x_{1} + 3x_{2} \leq 45 \quad \text{(기계 시간)}$$

$$x_{1} \leq 12 \quad \text{(마케팅 제약)}$$

### 표준형 (Standard Form)

슬랙 변수(Slack Variable)를 도입하여 등호 형태로 변환:

$$\max Z = 300x_{1} + 250x_{2} + 0s_{1} + 0s_{2} + 0s_{3}$$

$$2x_{1} + x_{2} + s_{1} = 40$$

$$x_{1} + 3x_{2} + s_{2} = 45$$

$$x_{1} + s_{3} = 12$$

- $s_1$: 작업자 시간 슬랙 ($s_1 \geq 0$)
- $s_2$: 기계 시간 슬랙 ($s_2 \geq 0$)
- $s_3$: 마케팅 제약 슬랙 ($s_3 \geq 0$)

## 풀이: 반복 1 — 초기 기저 실현 가능해 (Initial BFS at Vertex A)

### 초기 태블로 (Initial Simplex Tableau)

| z | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|---|-------|-------|-------|-------|-------|-----|
| 1 | -300  | -250  |   0   |   0   |   0   |  0  |
| 0 |   2   |   1   |   1   |   0   |   0   |  40 |
| 0 |   1   |   3   |   0   |   1   |   0   |  45 |
| 0 |   1   |   0   |   0   |   0   |   1   |  12 |

**초기 기저 실현 가능해 (Vertex A):**
$x_1=0, x_2=0, s_1=40, s_2=45, s_3=12$ → $Z = 0$

**최적성 판단:** 목적 함수 행에 음수 계수가 존재 ($-300, -250$) → 아직 최적이 아님.

**진입 변수(Entering Variable) 선택:** $x_1$ (계수 $-300$이 더 작아 $Z$ 증가율이 높음)

### 비율 검정 (Ratio Test)

| 기저 | $x_1$ 계수 | RHS | Ratio |
|------|-----------|-----|-------|
| $s_1$ | 2 | 40 | 40/2 = **20** |
| $s_2$ | 1 | 45 | 45/1 = 45 |
| $s_3$ | 1 | 12 | 12/1 = **12** ← 최소 |

**이탈 변수(Leaving Variable):** $s_3$ (Ratio 최소 = 12) → Vertex A에서 D로 이동

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams.update({'font.size': 14})

x = np.linspace(0, 100, 2000)
y1 = -2 * x + 40        # 2x1 + x2 <= 40
y2 = (45 - x) / 3.0     # x1 + 3x2 <= 45

plt.plot(x, y1, label=r'$2x_{1} + x_{2} \leq 40$')
plt.plot(x, y2, label=r'$x_{1} + 3x_{2} \leq 45$')
plt.axvline(x=12, label=r'$x_{1} \leq 12$', c='g')
plt.xlim((0, 50))
plt.ylim((0, 45))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')
y4 = np.minimum(y1, y2)
plt.fill_between(x, y4, 0, where=x < 12, color='grey', alpha=0.5)
plt.annotate('A (초기해)', xy=(0, 0), xytext=(1, 1), color='blue')
plt.annotate('B', xy=(0, 15), xytext=(0.5, 16))
plt.annotate('D', xy=(12, 0), xytext=(12.3, 1))
plt.annotate('E', xy=(12, 11), xytext=(12.3, 12))
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('초기 가능 영역 — Vertex A에서 시작')
plt.show()

## 풀이: 반복 2 — Vertex A → D (피벗 연산)

### 피벗 연산 (Pivot Operation)

피벗 원소: $s_3$ 행의 $x_1$ 계수 = 1 (이미 1이므로 정규화 불필요)

다른 행에서 $x_1$ 계수를 0으로 만드는 행 연산:

| 기저 | z | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS  |
|------|---|-------|-------|-------|-------|-------|------|
| -    | 1 |   0   | -250  |   0   |   0   |  300  | 3600 |
| $s_1$| 0 |   0   |   1   |   1   |   0   |  -2   |  16  |
| $s_2$| 0 |   0   |   3   |   0   |   1   |  -1   |  33  |
| $x_1$| 0 |   1   |   0   |   0   |   0   |   1   |  12  |

**Vertex D의 해:** $x_1=12, x_2=0, s_1=16, s_2=33, s_3=0$ → $Z = 3600$

**최적성 판단:** $x_2$ 계수 $-250 < 0$ → 아직 최적이 아님.

### 비율 검정 (두 번째)

| 기저   | $x_2$ 계수 | RHS | Ratio |
|--------|-----------|-----|-------|
| $s_1$  | 1 | 16 | 16/1 = **16** ← 최소 |
| $s_2$  | 3 | 33 | 33/3 = 11 |

**진입:** $x_2$, **이탈:** $s_2$ (Ratio 최소) → Vertex D에서 E로 이동

## 풀이: 반복 3 — Vertex D → E (최적해)

### 최종 태블로

피벗 원소: $s_2$ 행의 $x_2$ 계수 = 3 → 행을 3으로 나눈 뒤 다른 행에서 소거:

| 기저   | z | $x_1$ | $x_2$ | $s_1$ | $s_2$  | $s_3$  | RHS  |
|--------|---|-------|-------|-------|--------|--------|------|
| -      | 1 |   0   |   0   |   0   | 250/3  | 25/3   | 6350 |
| $s_1$  | 0 |   0   |   0   |   1   | -1/3   | -5/3   |  5   |
| $x_2$  | 0 |   0   |   1   |   0   |  1/3   | -1/3   |  11  |
| $x_1$  | 0 |   1   |   0   |   0   |   0    |   1    |  12  |

**최적해 (Vertex E):** $x_1=12, x_2=11, s_1=5, s_2=0, s_3=0$ → $Z_{max} = 6350$

**최적성 확인:** 목적 함수 행 모든 계수 $\geq 0$ → **최적해 확정**

**경영 해석:** $P_1$ 12개, $P_2$ 11개 생산 시 하루 수익 **6,350유로** 최대화

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams.update({'font.size': 14})

x = np.linspace(0, 100, 2000)
y1 = -2 * x + 40
y2 = (45 - x) / 3.0
y5 = (6350 - 300 * x) / 250  # 최적 목적 함수 등고선 Z=6350

plt.plot(x, y1, label=r'$2x_{1} + x_{2} \leq 40$')
plt.plot(x, y2, label=r'$x_{1} + 3x_{2} \leq 45$')
plt.axvline(x=12, label=r'$x_{1} \leq 12$', c='g')
plt.plot(x, y5, label=r'$Z=6350$ (최적 등고선)', linestyle='--', color='red')
plt.xlim((0, 50))
plt.ylim((0, 45))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')
y4 = np.minimum(y1, y2)
plt.fill_between(x, y4, 0, where=x < 12, color='grey', alpha=0.5)
# 꼭짓점 표시
for pt, label, offset in [((0,0),'A',(0.5,0.5)), ((0,15),'B',(0.5,15.5)),
                            ((12,0),'D',(12.3,0.5)), ((12,11),'E ★최적',(12.3,11.5))]:
    plt.scatter(*pt, zorder=5, color='blue' if '★' not in label else 'red')
    plt.annotate(label, xy=pt, xytext=offset,
                 color='red' if '★' in label else 'black')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title(r'최적해 Vertex E: $x_1=12, x_2=11, Z=6350$')
plt.show()

## 결론 (Conclusion)

심플렉스 알고리즘 반복 요약:

- **반복 0 (Vertex A):** $Z = 0$ → 비최적 (음수 계수 존재)
- **반복 1 (Vertex D):** $x_1$ 진입, $s_3$ 이탈 → $Z = 3{,}600$
- **반복 2 (Vertex E):** $x_2$ 진입, $s_2$ 이탈 → $Z = 6{,}350$ ✓ **최적**

**핵심 원리:**
1. **진입 변수 선택:** 목적 함수 계수가 가장 작은(음수) 변수
2. **이탈 변수 선택:** 비율 검정(Ratio Test)에서 최소 비율 행
3. **피벗 연산:** 진입 변수 열을 단위 벡터로 만드는 행 연산
4. **종료 조건:** 목적 함수 행 모든 계수 $\geq 0$

> 그래픽으로는 꼭짓점 이동(A → D → E), 대수적으로는 기저 변수 교체가 동일한 과정을 나타냅니다.